# Module 4: Applying Standards in Practice — Harmonizing Multi-Source Regional Data

**Unit B · Week 4** · Track 1 — Data Integration, Standards, Metadata & Quality

The first module where all three of this track's raw sources meet: joined on P-codes rather than names, dates normalized to ISO 8601, and a HXL tag row added to the published intermediate file.

## Learning objectives

- Add HXL hashtags to a raw dataset.
- Join disparate regional datasets using P-codes instead of place names.
- Normalize dates and country/district codes to ISO standards, and validate join coverage rather than silently dropping unmatched rows.


## Setup

This notebook reads the raw practice files in `../../../data/raw/`, built by
`data/make_track1_sources.py` — three partner-style exports shaped like a
real regional hub's source-system landscape (an ERP/financial export, a
survey-platform export, and an HDX-style pull), plus the P-code gazetteer
used to reconcile them. All four are **synthetic**; see `data/README.md`.
Run `python3 data/make_track1_sources.py` once from the repo root before
working through this notebook if those files aren't there yet.

## Lesson content

- Adding a HXL hashtag row to a dataset programmatically, and reading HXL-tagged data back in a way that's robust to column reordering (because the hashtag, not the column position, carries the meaning).
- Joining on P-codes instead of district names: looking up the correct P-code for each row from the gazetteer, then joining as an exact key, eliminating the silent mismatches that name-based joins produce (the same failure mode flagged, but not yet solved, in the Track 2 mapping module).
- Normalizing dates to ISO 8601 before any merge, so a later join doesn't fail because one source used `01/02/2025` and another used `2-Jan-25` — exactly the two formats Partner A and Partner B use here.
- Validating coverage after every harmonization step: how many rows failed to match a P-code, and why. A harmonized dataset that silently drops unmatched rows is a data-quality problem masquerading as a successful join.

In [1]:
import pandas as pd

gaz = pd.read_csv("../../../data/raw/cod_ab_gazetteer.csv")

a = pd.read_csv("../../../data/raw/partner_a_finance_export.csv")
b = pd.read_csv("../../../data/raw/partner_b_survey_export.csv")

# --- Coverage BEFORE any correction: how many rows fail to match a
#     district name exactly as the source spelled it? ---
unmatched_a_raw = a[~a["reg"].isin(gaz["district_name"])]
unmatched_b_raw = b[~b["region_name"].isin(gaz["district_name"])]
print(f"Partner A: {len(unmatched_a_raw)} row(s) unmatched before correction")
print(f"Partner B: {len(unmatched_b_raw)} row(s) unmatched before correction")
if len(unmatched_a_raw):
    print("  e.g.", unmatched_a_raw['reg'].unique())
if len(unmatched_b_raw):
    print("  e.g.", unmatched_b_raw['region_name'].unique())

Partner A: 1 row(s) unmatched before correction
Partner B: 1 row(s) unmatched before correction
  e.g. <StringArray>
['Nyarugenge Dist.']
Length: 1, dtype: str
  e.g. <StringArray>
['Rwamagana ']
Length: 1, dtype: str


In [2]:
# --- Fix: a small, reviewed alias map for known spelling variants,
#     plus whitespace stripping. Flagged and corrected, never dropped. ---
ALIASES = {"Nyarugenge Dist.": "Nyarugenge", "Rwamagana ": "Rwamagana"}

a["reg"] = a["reg"].str.strip().replace(ALIASES)
b["region_name"] = b["region_name"].str.strip().replace(ALIASES)

still_unmatched = pd.concat([
    a[~a["reg"].isin(gaz["district_name"])],
    b[~b["region_name"].isin(gaz["district_name"])],
])
print(f"Still unmatched after alias correction: {len(still_unmatched)} row(s)")

Still unmatched after alias correction: 0 row(s)


In [3]:
# --- Normalize both sources' dates to ISO 8601 and attach P-codes ---
a["date"] = pd.to_datetime(a["date"], format="%d/%m/%Y")
b["date"] = pd.to_datetime(b["collection_date"], format="%d-%b-%y")

a = a.merge(gaz, left_on="reg", right_on="district_name", how="left")
b = b.merge(gaz, left_on="region_name", right_on="district_name", how="left")

print("Partner A date range:", a["date"].min().date(), "to", a["date"].max().date())
a[["district_name", "district_pcode", "date", "rev"]].head()

Partner A date range: 2023-01-01 to 2024-12-01


,district_name,district_pcode,date,rev
0,Gasabo,SIM-GAS,2023-01-01,39.37
1,Gasabo,SIM-GAS,2023-02-01,45.24
2,Gasabo,SIM-GAS,2023-03-01,41.16
3,Gasabo,SIM-GAS,2023-04-01,42.91
4,Gasabo,SIM-GAS,2023-05-01,42.05


In [4]:
# --- Add a HXL tag row and publish the harmonized intermediate file ---
harmonized = a[["district_name", "district_pcode", "date", "rev"]].rename(
    columns={"district_name": "district", "rev": "value"}
)
harmonized["date"] = harmonized["date"].dt.strftime("%Y-%m-%d")

hxl_row = pd.DataFrame([{
    "district": "#adm2+name", "district_pcode": "#adm2+code",
    "date": "#date", "value": "#value+funding",
}])
hxl_tagged = pd.concat([hxl_row, harmonized], ignore_index=True)
hxl_tagged.to_csv("../../../data/processed/harmonized_regional_data.csv", index=False)
print(f"Wrote {len(harmonized):,} rows (plus 1 HXL tag row) to harmonized_regional_data.csv")

Wrote 240 rows (plus 1 HXL tag row) to harmonized_regional_data.csv


## Your turn

Confirm both misspelled rows (one in Partner A, one in Partner B) are caught by the before/after unmatched-row check, then extend the harmonization to also bring in Partner C's `indicator` and `outcome` columns, joined on `district_pcode` and `date`.

**Formative assessment.** Submitted harmonized, HXL-tagged CSV, graded on join correctness (unmatched rows flagged, not dropped), correct hashtags, and ISO-format dates.